# Wikidata Airport Dataset — EDA

Exploratory analysis of `data/interim/wikidata_airports.csv`, the fused, conflict-resolved dataset produced by `wikidata_extraction.ipynb`. This is about the *final* dataset's shape, distributions, and anomalies — for how each attribute was sourced/profiled/resolved, see `wikidata_profiling.ipynb` and `wikidata_extraction.ipynb`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

df = pd.read_csv(project_root / "data" / "interim" / "wikidata_airports.csv")
df.shape

(24237, 13)

## Overview

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24237 entries, 0 to 24236
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   airport     24237 non-null  object 
 1   name        24237 non-null  object 
 2   country     24221 non-null  object 
 3   elevation   18212 non-null  float64
 4   faa         10950 non-null  object 
 5   iata        7418 non-null   object 
 6   icao        10979 non-null  object 
 7   located_in  23878 non-null  object 
 8   operator    2656 non-null   object 
 9   owner       309 non-null    object 
 10  timezone    10004 non-null  object 
 11  lat         24189 non-null  float64
 12  lon         24189 non-null  float64
dtypes: float64(3), object(10)
memory usage: 2.4+ MB


In [3]:
df.head()

,airport,name,country,elevation,faa,iata,icao,located_in,operator,owner,timezone,lat,lon
0,http://www.wikidata.org/entity/Q1000018,Ji'an Jinggangshan Airport,People's Republic of China,86.0000,NaN,JGS,ZSGS,Taihe County,NaN,NaN,NaN,26.857500,114.737222
1,http://www.wikidata.org/entity/Q1000025,Changde Taohuayuan Airport,People's Republic of China,NaN,NaN,CGD,ZGCD,Hunan,NaN,NaN,NaN,28.918900,111.640000
2,http://www.wikidata.org/entity/Q1002604,Jiujiang Lushan Airport,People's Republic of China,49.9872,NaN,JIU,ZSJJ,Jiujiang,NaN,NaN,NaN,29.476944,115.801111
3,http://www.wikidata.org/entity/Q1006478,Aksu Hongqipo Airport,People's Republic of China,1163.0000,NaN,AKU,ZWAK,Hongqipo,NaN,NaN,NaN,41.262500,80.291667
4,http://www.wikidata.org/entity/Q1006900,Mohe Gulian Airport,People's Republic of China,579.0000,NaN,OHE,ZYMH,Heilongjiang,NaN,NaN,NaN,52.921111,122.420556


## Missingness

In [4]:
(df.isna().mean().sort_values(ascending=False) * 100).round(2).rename("missing_%")

owner         98.73
operator      89.04
iata          69.39
timezone      58.72
faa           54.82
icao          54.70
elevation     24.86
located_in     1.48
lat            0.20
lon            0.20
country        0.07
airport        0.00
name           0.00
Name: missing_%, dtype: float64

## Uniqueness / duplicate checks

One row per `airport` URI is expected by construction (it's the groupby key in the extraction notebook) — this checks whether that key is actually unique, and whether any (name, lat, lon) combination repeats, which would suggest the same real-world airport appearing under two different Wikidata items.

In [5]:
print("duplicate airport URIs:", df["airport"].duplicated().sum())

dup_name_coords = df[df.duplicated(subset=["name", "lat", "lon"], keep=False) & df["lat"].notna()]
print("rows sharing the same (name, lat, lon) with another row:", len(dup_name_coords))
dup_name_coords.sort_values("name").head(10)

duplicate airport URIs: 0
rows sharing the same (name, lat, lon) with another row: 2


,airport,name,country,elevation,faa,iata,icao,located_in,operator,owner,timezone,lat,lon
1861,http://www.wikidata.org/entity/Q130710790,Rafael Hernández Airport,United States,NaN,NaN,NaN,NaN,Puerto Rico,NaN,NaN,UTC−04:00,18.495,-67.129444
5795,http://www.wikidata.org/entity/Q2276582,Rafael Hernández Airport,United States,72.2376,BQN,BQN,TJBQ,Aguadilla,NaN,NaN,NaN,18.495,-67.129444


## Numeric attributes

`elevation` (metres) and `lat`/`lon` (decimal degrees). Coordinates should fall within [-90, 90] / [-180, 180] by construction (GeoSPARQL extraction), but elevation has no such hard bound — flagging physically implausible values (below the Dead Sea shore at ~-430 m, or above the highest paved airstrips at ~5,500 m) is a cheap sanity check for bad data.

In [6]:
df[["elevation", "lat", "lon"]].describe()

,elevation,lat,lon
count,18212.000000,24189.000000,24189.000000
mean,368.925562,29.453696,-50.208390
std,469.579215,23.700320,76.513052
min,-378.000000,-79.250000,-179.876945
25%,65.477520,26.733611,-96.575630
50%,227.000000,37.140246,-82.662222
75%,449.656200,43.093083,-1.789440
max,4411.000000,82.517778,179.975500


In [7]:
implausible_elevation = df[(df["elevation"] < -430) | (df["elevation"] > 5500)]
print(f"{len(implausible_elevation)} airports with a physically implausible elevation")
implausible_elevation[["airport", "name", "elevation"]].sort_values("elevation", ascending=False).head(10)

0 airports with a physically implausible elevation


,airport,name,elevation


## Categorical / identifier attributes

In [8]:
df["country"].value_counts().head(15)

country
United States                       13928
Canada                               1394
Brazil                                657
Australia                             484
Russia                                382
Chile                                 352
People's Republic of China            322
Papua New Guinea                      287
Indonesia                             252
India                                 238
Democratic Republic of the Congo      207
United Kingdom                        197
France                                178
Argentina                             168
Bolivia                               162
Name: count, dtype: int64

In [9]:
for col in ["iata", "icao", "faa"]:
    print(f"{col}: {df[col].notna().sum()} airports have a value ({df[col].notna().mean():.1%})")

iata: 7418 airports have a value (30.6%)
icao: 10979 airports have a value (45.3%)
faa: 10950 airports have a value (45.2%)


## Next steps

This dataset is the Wikidata side of the integration. Next: repeat profiling + extraction for OurAirports and the Kaggle dataset, then move to schema mapping / identity resolution across all three (see `docs/idea_abstract.md`).